# Interaction Effect

In [29]:
import time
from sklearn import datasets
from sklearn import ensemble
from sklearn.model_selection import train_test_split
import numpy as np
import pandas as pd

import lightgbm as lgb

from mli.explanation.h_statistic import HStatistic


iris_data = datasets.load_iris()
X = iris_data.data
y = iris_data.target
print(f"Original data shape: {X.shape}")

Original data shape: (150, 4)


### Train and Test split

In [2]:
# Train and Test
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"Train set shape {X_train.shape}")
print(f"Test set shape {X_test.shape}")

num_train, num_feature = X_train.shape
print(f"Number of features: {num_feature}")
print("----------------------------------")
print(f"Name of the features: {iris_data.feature_names}")
print("----------------------------------")
print(f"Number of classes(multi-class problem): {np.unique(y)}")

Train set shape (120, 4)
Test set shape (30, 4)
Number of features: 4
----------------------------------
Name of the features: ['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']
----------------------------------
Number of classes(multi-class problem): [0 1 2]


### Modeling

In [3]:
# First just using lightgbm
lgb_train = lgb.Dataset(X_train, y_train, free_raw_data=False)
lgb_eval = lgb.Dataset(X_test, y_test, reference=lgb_train, free_raw_data=False)

# generate feature names
feature_name = iris_data.feature_names

# specify your configurations as a dict
params = {
    'boosting_type': 'gbdt',
    'objective': 'multiclass',
    'metric': 'multi_logloss', # since it's a multi-class problem
    'num_class':3,
    'num_leaves': 31,
    'learning_rate': 0.05,
    'feature_fraction': 0.9,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'verbose': 0
}

In [ ]:
print('Starting training...')
# feature_name and categorical_feature
gbm = lgb.train(params,
                lgb_train,
                num_boost_round=40,
                valid_sets=lgb_train,  # eval training data
                feature_name=feature_name)

In [5]:
from sklearn.metrics import mean_squared_error

y_pred = gbm.predict(X_test)
y_class = [np.argmax(x_i) for x_i in y_pred]

# eval with loaded model
print("The rmse of loaded model's prediction is:", mean_squared_error(y_test, np.array(y_class)) ** 0.5)

The rmse of loaded model's prediction is: 0.0


In [105]:
gbm.feature_importance(importance_type='split')

array([ 53,  25, 119,  94])

### Computing Interactions

In [38]:
def predict_prob(x, class_index=0):
    # Multi-class use-case
    return pd.DataFrame(gbm.predict(x)[:, class_index])

In [54]:
# train
X_train_df = pd.DataFrame(X_train, columns=feature_name)
# test
X_test_df = pd.DataFrame(X_test, columns=feature_name)

predict_prob(X_train_df.loc[0])

,0
0,0.862496


In [45]:
X_train_df = pd.DataFrame(X_train, columns=feature_name)
interactions = HStatistic("H-stat for pairs").explain(
            feature_name,
            X_train_df,
            predict_method=predict_prob)

In [46]:
print(interactions)


'('sepal length (cm)', 'sepal width (cm)')':
  'p_0':
    0
'('sepal length (cm)', 'petal length (cm)')':
  'p_0':
    0.0
'('sepal length (cm)', 'petal width (cm)')':
  'p_0':
    0.0
'('sepal width (cm)', 'petal length (cm)')':
  'p_0':
    0.0
'('sepal width (cm)', 'petal width (cm)')':
  'p_0':
    0.0
'('petal length (cm)', 'petal width (cm)')':
  'p_0':
    0.023793548278393303



## DAI Model

### 1. Connect to DAI instance

In [49]:
# Using the latest release DAI-1.5.1
# Step 1: Upload the data to DAI engine
import requests
import math
import os
import pandas as pd
from h2oai_client import Client, ModelParameters, InterpretParameters

In [ ]:
import os

ip = 'prerelease.h2o.ai'
username = 'h2oai'
# use environment variables or secure credential stores
password = os.getenv('H2O_PASSWORD', 'h2oai')  # Default for local testing only

h2oai = Client(address = 'http://' + ip, username=username, password=password)

# info about the DAI Client:
print(h2oai.address)

### 2. Upload dataset to Driverless AI

In [57]:
feature_list = feature_name + ['target']
print(feature_list)


# train
data_df_train = pd.DataFrame(data=np.c_[X_train, y_train],
                         columns=feature_list)
print(data_df_train.head(2))

# test
data_df_test = pd.DataFrame(data=np.c_[X_test, y_test],
                         columns=feature_list)
print(data_df_test.head(2))


# persist the dataframes
data_df_train.to_csv("iris_train.csv", index=False)
data_df_test.to_csv("iris_test.csv", index=False)

current_directory = os.getcwd()
print(current_directory)
train_path_dai = f"{current_directory}/iris_train.csv"
test_path_dai = f"{current_directory}/iris_test.csv"

# Upload datasets
train = h2oai.upload_dataset_sync(train_path_dai)
test = h2oai.upload_dataset_sync(test_path_dai)

['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)', 'target']
   sepal length (cm)  sepal width (cm)  petal length (cm)  petal width (cm)  \
0                4.3               2.0                1.0               0.2   
1                4.3               2.0                1.5               0.4   

   target  
0     0.0  
1     0.0  
   sepal length (cm)  sepal width (cm)  petal length (cm)  petal width (cm)  \
0                6.1               2.8                4.7               1.2   
1                5.7               3.8                1.7               0.3   

   target  
0     1.0  
1     0.0  
/home/ubuntu/h2o/mli-2


### 3. Set up parameters for Driverless AI experiment

In [ ]:
# Set the parameters you want to pass to DAI
dataset_key=train.key # Dataset to use for DAI
validset_key='' # Validation set to use for DAI (Note, we are not using one for this experiment)
testset_key=test.key # Test set to use for DAI
target="target" # Target column for DAI
# dropped_cols=["sample_weight"] #List of columns to drop. In this case we are dropping 'sample_weight'
weight_col=None # The column that indicates the per row observation weights.
                # If None, each row will have an observation weight of 1
fold_col=None # The column that indicates the fold. If None, the folds will be determined by DAI

is_time_series=False # Whether or not the experiment is a time series problem
is_classification=True # Inform DAI if the problem type is a classification (binomial/multinomial)
                    # or not (regression)

enable_gpus=True # Whether or not to enable GPUs
seed=1234 # Use seed for reproducibility
scorer_str='logloss' # Set evaluation metric. In this case we are interested in optimizing R-squared
accuracy=1 # Accuracy setting for experiment (One of the 3 knobs you see in the DAI UI)
time=1 # Time setting for experiment (One of the 3 knobs you see in the DAI UI)
interpretability=2 # Interpretability setting for experiment (One of the 3 knobs you see in the DAI UI)
config_overrides=None # Extra parameters that can be passed in TOML format

In [ ]:
experiment = h2oai.start_experiment_sync(
    #Datasets
    dataset_key=dataset_key,
    validset_key = validset_key,
    testset_key=testset_key,

    #Columns
    target_col=target,
    #cols_to_drop=dropped_cols,
    weight_col=weight_col,
    fold_col=fold_col,

    #Parameters
    is_classification=is_classification,
    enable_gpus=enable_gpus,
    seed=seed,
    accuracy=accuracy, # DAI suggested for accuracy
    time=time, # DAI suggested for time
    interpretability=interpretability, # DAI suggested for interpretability
    scorer=scorer_str,
    # Extra parameters that can be passed in TOML format
    config_overrides=None
)

#### Quick Summary:

In [65]:
print(f"Name of the experiment: {experiment.description}")

# Download Summary
import subprocess
summary_path = h2oai.download(src_path=experiment.summary_path, dest_dir=".")
dir_path = "./h2oai_experiment_summary_" + experiment.key
subprocess.call(['unzip', '-o', summary_path, '-d', dir_path], shell=False)

# View Features
features = pd.read_table(dir_path + "/features.txt", sep=',', skipinitialspace=True)
features.head()

Name of the experiment: lusehidi


,Relative Importance,Feature,Description
0,1.00000,9_InteractionAdd:petal length (cm):petal width...,[petal length (cm)] + [petal width (cm)]
1,0.76396,3_ClusterDist3:petal width (cm).1,Distances to cluster center after segmenting c...
2,0.43968,3_ClusterDist3:petal width (cm).2,Distances to cluster center after segmenting c...
3,0.35761,3_ClusterDist3:petal width (cm).0,Distances to cluster center after segmenting c...
4,0.17399,18_InteractionDiv:petal length (cm):petal widt...,[petal length (cm)] / [petal width (cm)]


### 4. Set up scoring package from Driverless AI experiment

In [67]:
h2oai.download(experiment.scoring_pipeline_path, '.')

'./scorer.zip'

In [83]:
# unpack
from shutil import unpack_archive
unpack_archive('scorer.zip', '.')

In [80]:
#Import scoring module
!python -m pip install scoring-pipeline/scoring_h2oai_experiment_*.whl

You are using pip version 9.0.3, however version 19.0.1 is available.
You should consider upgrading via the 'pip install --upgrade pip' command.


### 5. Computing interactions on DAI model

In [89]:
from scoring_h2oai_experiment_lusehidi import Scorer

In [90]:
%%capture
# Create a singleton Scorer instance.
# For optimal performance, create a Scorer instance once, and call score() or score_batch() multiple times.
scorer = Scorer()

In [92]:
import datatable as dt
def predict_dt(X):
    X = dt.Frame(X)
    return scorer.score_batch(X).iloc[:, 0]

In [93]:
interactions = HStatistic("H-stat for pairs on DAI models").explain(
            feature_name,
            X_train_df,
            predict_method=predict_dt)

In [94]:
print(interactions)


'('sepal length (cm)', 'sepal width (cm)')':
  'p_0':
    0
'('sepal length (cm)', 'petal length (cm)')':
  'p_0':
    0.0
'('sepal length (cm)', 'petal width (cm)')':
  'p_0':
    0.0
'('sepal width (cm)', 'petal length (cm)')':
  'p_0':
    0.0
'('sepal width (cm)', 'petal width (cm)')':
  'p_0':
    0.0
'('petal length (cm)', 'petal width (cm)')':
  'p_0':
    0.011393515088801218

